<a href="https://colab.research.google.com/github/kyungeun-l/DS_2/blob/main/%5B%ED%8C%8C%EC%9D%B4%EC%8D%AC_%EA%B8%B0%EC%B4%88%5D_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

배경

부동산 플랫폼의 매물 운영팀은 프로모션에 활용할 주택 후보 목록을 정리하고 있습니다.

운영팀은 먼저 전달받은 CSV 파일의 데이터 구조와 값의 자료형을 점검해야 합니다. 이후 계산 가능한 형태로 데이터를 변환하고, 유효하지 않은 데이터를 제외한 뒤 업무 조건에 맞는 주택 후보를 선별합니다. 마지막으로 주택별 크기당 가격을 계산할 수 있는 함수를 구현해 전체 데이터에 적용합니다.

이번 과제에서는 CSV 파일 읽기, 자료형 변환, 조건에 따른 데이터 선별, 함수와 예외 처리까지 이어지는 기본 데이터 점검 흐름을 구현합니다.

In [7]:
#1. CSV 파일 읽기

import pandas as pd
try:
    df_house=pd.read_csv('/content/1.01.house.price.csv')
except FileNotFoundError:
    print("파일명과 업로드 여부를 확인하세요.")
else:
    print(df_house.iloc[0])
    print(len(df_house))
    print(type(df_house.iloc[0]["price"]))

ID        1.0
size     34.0
year      5.0
price     5.0
Name: 0, dtype: float64
10
<class 'numpy.float64'>


In [12]:
# 2. 문자열을 숫자로 변환하기

houses = []

for i in range(len(df_house)):
    if df_house.iloc[i].isnull().sum() > 0:
        print(f"{i+1}:필수 값 누락")
        continue
    try:
        houses.append({
            "ID": int(df_house.iloc[i]["ID"]),
            "size": int(df_house.iloc[i]["size"]),
            "year": int(df_house.iloc[i]["year"]),
            "price": float(df_house.iloc[i]["price"])
        })
    except (ValueError, TypeError):
        print(f"{i+1}: 숫자 변환 불가")

print("정상 저장 행 수:", len(houses))
print("제외 행 수:", len(df_house)-len(houses))
print(houses)

정상 저장 행 수: 10
제외 행 수: 0
[{'ID': 1, 'size': 34, 'year': 5, 'price': 5.0}, {'ID': 2, 'size': 25, 'year': 5, 'price': 2.5}, {'ID': 3, 'size': 30, 'year': 2, 'price': 4.0}, {'ID': 4, 'size': 38, 'year': 20, 'price': 3.0}, {'ID': 5, 'size': 44, 'year': 12, 'price': 3.3}, {'ID': 6, 'size': 48, 'year': 18, 'price': 4.2}, {'ID': 7, 'size': 52, 'year': 22, 'price': 4.6}, {'ID': 8, 'size': 60, 'year': 19, 'price': 6.0}, {'ID': 9, 'size': 34, 'year': 18, 'price': 3.0}, {'ID': 10, 'size': 34, 'year': 22, 'price': 2.9}]


In [20]:
# 3. 조건에 맞는 주택 찾기

houses_test = houses.copy()

houses_test.extend([
    {"ID": 101, "year": 5, "price": 3},
    {"ID": 102, "size": 35, "year": 5, "price": None},
    {"ID": 103, "size": "오류", "year": 5, "price": 3},
    {"ID": 104, "size": 35, "year": 5, "price": 3}
])

selected = []

for house in houses_test:
    if "size" not in house or "price" not in house:
        print(f"ID {house['ID']} 제외: 키 없음")
        continue

    if pd.isnull(house["size"]) or pd.isnull(house["price"]):
        print(f"ID {house['ID']} 제외: 결측값")
        continue
    try:
        size = float(house["size"])
        price = float(house["price"])

    except (ValueError, TypeError):
        print(f"ID {house['ID']} 제외: 숫자 변환 불가")
        continue

    if size <= 0 or price < 0:
        print(f"ID {house['ID']} 제외: 유효하지 않은 값")
        continue

    if size >= 30 and price <= 4:
        selected.append(house)

for house in selected:
    print(house["ID"])

print("선택된 주택 수:", len(selected))

ID 101 제외: 키 없음
ID 102 제외: 결측값
ID 103 제외: 숫자 변환 불가
3
4
5
9
10
104
선택된 주택 수: 6


In [26]:
# 4. 함수와 예외 처리

def unit_price(price, size):
    try:
        price = float(price)
        size = float(size)

        if size <= 0:
            print("크기 오류: 0보다 커야 합니다.")
            return None

        if price < 0:
            print("가격 오류: 0보다 커야 합니다.")
            return None
        return price / size

    except ValueError:
        print("값 오류: 숫자로 변환할 수 없는 문자열입니다.")
        return None

    except TypeError:
        print("자료형 오류: 숫자 또는 숫자 형태의 문자열을 입력하세요.")
        return None

# 전체 houses에 적용
for house in houses:
    result = unit_price(house["price"], house["size"])
    print(f"ID {house['ID']} : {result}")

# 순서대로 실행하여 반환값과 안내 문구 확인
tests = [
    (4, 40),
    ("4", "40"),
    (4, 0),
    (-4, 40),
    ("오류", 40),
    (None, 40),
    (6, 30)
]
for price, size in tests:
    print(unit_price(price, size))

ID 1 : 0.14705882352941177
ID 2 : 0.1
ID 3 : 0.13333333333333333
ID 4 : 0.07894736842105263
ID 5 : 0.075
ID 6 : 0.08750000000000001
ID 7 : 0.08846153846153845
ID 8 : 0.1
ID 9 : 0.08823529411764706
ID 10 : 0.08529411764705883
0.1
0.1
크기 오류: 0보다 커야 합니다.
None
가격 오류: 0보다 커야 합니다.
None
값 오류: 숫자로 변환할 수 없는 문자열입니다.
None
자료형 오류: 숫자 또는 숫자 형태의 문자열을 입력하세요.
None
0.2


In [29]:
# 5. 목적에 맞는 자료구조 선택하기

# 5-1. 전체 주택 ID를 원래 순서대로 저장하고 출력
# 리스트[]: 입력 순서 유지하기 때문
house_ids = []
for house in houses:
    house_ids.append(house["ID"])
print(house_ids)

# 5-2. 주택의 year 값을 중복 없이 저장하고 출력
# 집합: 순서는 보장하지 않으나 중복된 값 자동으로 제거함
years = set()
for house in houses:
    years.add(house["year"])
print(years)

# 5-3. 주택 ID로 가격을 조회할 수 있도록 저장하고, ID 3의 가격을 출력.
# 딕셔너리: ID를 키로 사용하여 가격을 바로 조회할 수 있음
price_by_id = {house["ID"]: house["price"] for house in houses}
print(price_by_id[3])

#5-4. 최소 크기 30과 최대 가격 4를 각각 하나의 원소로 갖는 묶음으로 저장
#튜플, 튜플은 생성 후 추가 또는 삭제하거나 다른 값으로 변경할 수 없기 때문.
conditions = (30, 4)
print(conditions)


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
{2, 5, 12, 18, 19, 20, 22}
4.0
(30, 4)
